In [2]:
!pip install sqlalchemy


   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ------------------- -------------------- 1.0/2.1 MB 12.5 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 10.9 MB/s  0:00:00

   ---------------------------------------- 0/2 [greenlet]
   ---------------------------------------- 0/2 [greenlet]
   ---------------------------------------- 0/2 [greenlet]
   ---------------------------------------- 0/2 [greenlet]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalchemy]
   -------------------- ------------------- 1/2 [sqlalc


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
import json
import pandas as pd
import sqlite3
from sqlalchemy import create_engine

with open("../data/raw/watch-history.json", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df["time"] = pd.to_datetime(df["time"], format="mixed", utc=True)

def safe_json(x):
    if isinstance(x, list):
        return json.dumps(x)
    return x
    
list_columns = ['subtitles', 'products', 'activityControls', 'details']
for col in list_columns:
    if col in df.columns:
        df[col] = df[col].apply(safe_json)

db_path = "../data/processed/media_analytics_2026_jan.sqlite"
engine = create_engine(f'sqlite:///{db_path}')

df.to_sql('watch_history', engine, if_exists='replace', index=False)

jan_2026_query = """
SELECT * FROM watch_history 
WHERE time >= '2026-01-01' AND time < '2026-02-01'
"""
jan_2026 = pd.read_sql(jan_2026_query, engine)

print(jan_2026.head())


    header                                          title  \
0  YouTube                         ぱんちどらいくどうーまん2話 を視聴しました   
1  YouTube              【神回】バイトやらかし2022【やらかしすぎ注意】 を視聴しました   
2  YouTube  【バイトやらかし】みんなのやらかしが止まらないので2回目やっちゃいますSP を視聴しました   
3  YouTube   【神回】みんなのバイトやらかし話が面白すぎて全国民必見だと思う【丸山礼】 を視聴しました   
4  YouTube                   【カラオケ】人生いろいろ / 島倉千代子 を視聴しました   

                                      titleUrl  \
0  https://www.youtube.com/watch?v=z95G5woMo_Y   
1  https://www.youtube.com/watch?v=pSWkhDtZp30   
2  https://www.youtube.com/watch?v=d7sRBjivaqs   
3  https://www.youtube.com/watch?v=H8rHrGSLSOQ   
4  https://www.youtube.com/watch?v=CqQCWD2MDT8   

                                           subtitles  \
0  [{"name": "K", "url": "https://www.youtube.com...   
1  [{"name": "\u4e38\u5c71\u793c\u30c1\u30e3\u30f...   
2  [{"name": "\u4e38\u5c71\u793c\u30c1\u30e3\u30f...   
3  [{"name": "\u4e38\u5c71\u793c\u30c1\u30e3\u30f...   
4  [{"name": "\u30ab\u30e9\u30aa\u30b1\u6b4c\u306...